# Evaluación del Sistema de Compliance Agents
## FinServ LATAM — XQuad Challenge
Notebook para medir la solidez del razonamiento de los agentes usando múltiples estrategias:
1. Invariance Testing — la decisión no cambia con variaciones irrelevantes
2. Adversarial Testing — el sistema resiste manipulaciones regulatorias críticas  
3. LLM-as-Judge — Claude evalúa la calidad del razonamiento de los agentes
4. Regulatory Coverage — los artículos correctos se recuperan para cada tipo de alerta

In [ ]:
import sys, os, asyncio
sys.path.insert(0, os.path.abspath('..'))

from agents import investigador, risk_analyzer, decision_agent
from data.models import DecisionType
from tools.llm_client import get_llm_client
from graph.pipeline import run_pipeline

print("Sistema cargado. LLM activo:", type(get_llm_client()).__name__)

## Test 1: Invariance Testing
La misma alerta procesada múltiples veces debe producir decisiones consistentes.
Un sistema de razonamiento sólido no debe ser inestable — variaciones aleatorias en la temperatura del LLM no deben cambiar la decisión final para casos claros (score muy alto o muy bajo).

In [ ]:
async def run_invariance_test(alert_id: str, n_runs: int = 3):
    """Corre el mismo alert N veces y verifica consistencia de la decisión."""
    results = []
    for i in range(n_runs):
        result = await run_pipeline(alert_id)
        results.append({
            "run": i + 1,
            "risk_score": result["risk_analysis"].risk_score,
            "decision": result["decision"].decision.value,
            "confidence": result["decision"].confidence,
        })
    
    decisions = [r["decision"] for r in results]
    scores = [r["risk_score"] for r in results]
    
    is_consistent = len(set(decisions)) == 1
    score_variance = max(scores) - min(scores)
    
    print(f"\n=== Invariance Test: {alert_id} ===")
    for r in results:
        print(f"  Run {r['run']}: score={r['risk_score']}/10, decision={r['decision']}, confidence={r['confidence']:.0%}")
    print(f"\nDecisión consistente: {'✅ SÍ' if is_consistent else '❌ NO'}")
    print(f"Varianza de score: {score_variance} puntos (aceptable si <= 2)")
    
    return {"consistent": is_consistent, "score_variance": score_variance, "results": results}

# Ejecutar para el caso de alto riesgo (debe ser siempre ESCALATE)
result_001 = asyncio.run(run_invariance_test("ALERT-001", n_runs=2))

## Test 2: Adversarial Testing — Reglas No Negociables
Verificamos que las reglas regulatorias críticas son inmunes a variaciones del LLM.
La regla PEP es hardcoded — el LLM nunca puede descartarla. Este test verifica esa invariante.

In [ ]:
def test_pep_override_invariant():
    """La regla PEP debe escapar siempre, independientemente del score de riesgo."""
    pep_context, trail = investigador.run("ALERT-PEP-003")
    
    # Forzar un score bajo artificialmente para simular "LLM equivocado"
    from data.models import RiskAnalysis
    low_risk_analysis = RiskAnalysis(
        alert_id="ALERT-PEP-003",
        risk_score=2,  # Score artificialmente bajo — el LLM "se equivocó"
        risk_justification="Score bajo simulado para test adversarial",
        anomalous_patterns=[],
        analyst_summary="Test adversarial: score forzado a 2/10"
    )
    
    decision, _ = decision_agent.run(pep_context, low_risk_analysis, trail)
    
    print("=== Test Adversarial: PEP Override ===")
    print(f"Score de riesgo inyectado: 2/10 (artificialmente bajo)")
    print(f"Decisión resultante: {decision.decision.value}")
    print(f"PEP override activo: {decision.is_pep_override}")
    print(f"Confianza: {decision.confidence:.0%}")
    print()
    
    assert decision.decision == DecisionType.ESCALATE, "❌ FALLO CRÍTICO: PEP no escaló"
    assert decision.is_pep_override is True, "❌ FALLO: pep_override debe ser True"
    assert decision.confidence == 1.0, "❌ FALLO: confianza debe ser 1.0 para PEP"
    print("✅ Test adversarial PASADO: La regla PEP es inviolable")

test_pep_override_invariant()

## Test 3: LLM-as-Judge — Evaluación de Solidez del Razonamiento
Usamos Claude como evaluador externo para medir si el razonamiento del agente es:
- **Faithful**: la justificación es consistente con los datos del caso
- **Regulatory**: las referencias regulatorias son correctas para el país/tipo de alerta
- **Complete**: el audit trail cubre todos los pasos necesarios para una auditoría
- **Proportional**: el score de riesgo es proporcional a la severidad de los indicadores

Esta técnica (LLM-as-Judge) es el estándar emergente para evaluar sistemas de razonamiento donde no existe un ground truth único.

In [ ]:
JUDGE_SYSTEM_PROMPT = """Eres un auditor senior de compliance financiero con 20 años de experiencia
en regulación LATAM (UIAF Colombia, CNBV México, SBS Perú).

Tu tarea es evaluar la calidad del razonamiento de un sistema de IA de compliance.
Debes producir una evaluación en JSON con exactamente estos campos:
- faithfulness_score: 1-5 (¿la justificación es consistente con los datos del caso?)
- regulatory_accuracy: 1-5 (¿las referencias regulatorias son correctas?)
- completeness_score: 1-5 (¿el audit trail cubre todos los pasos de una auditoría?)
- proportionality_score: 1-5 (¿el score de riesgo es proporcional a los indicadores?)
- overall_score: 1-5 (evaluación global)
- critique: string con observaciones del auditor (máx 150 palabras)
- would_pass_audit: boolean (¿pasaría una auditoría regulatoria real?)

Sé estricto. En compliance, un razonamiento mediocre puede tener consecuencias legales."""

def evaluate_with_llm_judge(alert_id: str):
    """Evalúa la calidad del razonamiento usando Claude como juez."""
    import asyncio, json
    
    result = asyncio.run(run_pipeline(alert_id))
    analysis = result["risk_analysis"]
    decision = result["decision"]
    
    # Preparar el contexto para el juez
    evaluation_context = f"""
CASO A EVALUAR: {alert_id}

ANÁLISIS DE RIESGO PRODUCIDO POR EL SISTEMA:
- Score: {analysis.risk_score}/10
- Justificación: {analysis.risk_justification}
- Patrones detectados: {len(analysis.anomalous_patterns)}
- Resumen analista: {analysis.analyst_summary}

DECISIÓN FINAL:
- Decisión: {decision.decision.value}
- Confianza: {decision.confidence:.0%}
- PEP override: {decision.is_pep_override}
- Resumen: {decision.final_summary}

AUDIT TRAIL ({len(decision.reasoning_chain)} pasos):
{chr(10).join([f"  Paso {s.step} [{s.agent}] {s.action}: {s.reasoning[:100]}" for s in decision.reasoning_chain])}

REFERENCIAS REGULATORIAS CITADAS:
{chr(10).join([f"  - {r.article} ({r.body.value}): {r.description[:80]}" for r in decision.regulatory_references])}
"""
    
    llm = get_llm_client()
    evaluation = llm.complete_json(JUDGE_SYSTEM_PROMPT, evaluation_context)
    
    print(f"\n{'='*60}")
    print(f"LLM-as-Judge: {alert_id}")
    print(f"{'='*60}")
    print(f"Faithfulness      : {'⭐'*evaluation['faithfulness_score']} ({evaluation['faithfulness_score']}/5)")
    print(f"Regulatory Acc.   : {'⭐'*evaluation['regulatory_accuracy']} ({evaluation['regulatory_accuracy']}/5)")
    print(f"Completeness      : {'⭐'*evaluation['completeness_score']} ({evaluation['completeness_score']}/5)")
    print(f"Proportionality   : {'⭐'*evaluation['proportionality_score']} ({evaluation['proportionality_score']}/5)")
    print(f"Overall           : {'⭐'*evaluation['overall_score']} ({evaluation['overall_score']}/5)")
    print(f"Pasa auditoría    : {'✅ SÍ' if evaluation['would_pass_audit'] else '❌ NO'}")
    print(f"\nCrítica del auditor:\n  {evaluation['critique']}")
    
    return evaluation

# Evaluar el caso de mayor riesgo
eval_001 = evaluate_with_llm_judge("ALERT-001")

In [ ]:
# Evaluar también el caso de bajo riesgo
eval_002 = evaluate_with_llm_judge("ALERT-002")

## Test 4: Cobertura Regulatoria del RAG
Para cada tipo de alerta, verificamos que el sistema RAG recupera los artículos
que son OBLIGATORIOS según la regulación del país correspondiente.

Ground truth definido por tipo de alerta:
- Estructuración (UIAF): debe incluir Art.15 (estructuración) y Art.8 (umbrales)
- PEP (SBS): debe incluir Art.18 (obligaciones PEP)
- Patrón inusual (CNBV): debe incluir D-110 (operaciones inusuales)

In [ ]:
def test_regulatory_coverage():
    """Verifica que el RAG recupera los artículos obligatorios para cada tipo de alerta."""
    from rag.retriever import HybridRetriever
    
    # Ground truth: artículos que DEBEN aparecer para cada consulta
    test_cases = [
        {
            "query": "estructuración smurfing transferencias fraccionadas umbral UIAF Colombia",
            "expected_keywords": ["structuring", "estructuración", "Art", "umbral", "50,000"],
            "alert_type": "structuring",
            "regulator": "UIAF"
        },
        {
            "query": "persona políticamente expuesta PEP monitoreo reforzado SBS Peru",
            "expected_keywords": ["PEP", "políticamente", "monitoreo", "reforzado"],
            "alert_type": "pep",
            "regulator": "SBS"
        },
        {
            "query": "operación inusual patrón atípico cliente México CNBV",
            "expected_keywords": ["inusual", "patrón", "operación"],
            "alert_type": "unusual_pattern",
            "regulator": "CNBV"
        },
    ]
    
    retriever = HybridRetriever()
    
    for case in test_cases:
        results = retriever.retrieve(case["query"], top_k=5)
        
        # Verificar cobertura: ¿aparecen las keywords obligatorias?
        all_content = " ".join([r.get("content", "") for r in results]).lower()
        hits = [kw for kw in case["expected_keywords"] if kw.lower() in all_content]
        coverage = len(hits) / len(case["expected_keywords"])
        
        print(f"\nAlerta tipo: {case['alert_type']} ({case['regulator']})")
        print(f"  Artículos recuperados: {len(results)}")
        print(f"  Keywords encontradas: {hits}")
        print(f"  Cobertura regulatoria: {coverage:.0%} {'✅' if coverage >= 0.6 else '⚠️'}")
        
        if results:
            print(f"  Top resultado: {results[0].get('article_id', 'N/A')} - score {results[0].get('score', 0):.3f}")

test_regulatory_coverage()

## Resumen de la Evaluación

| Dimensión | Método | Resultado esperado |
|---|---|---|
| Consistencia | Invariance testing (3 runs) | Misma decisión en 100% de casos claros |
| Reglas críticas | Adversarial testing | PEP siempre escala, score <= 3 siempre descarta |
| Calidad razonamiento | LLM-as-Judge | Overall >= 4/5, would_pass_audit = True |
| Cobertura RAG | Ground truth keywords | >= 60% de keywords obligatorias recuperadas |

### Limitaciones reconocidas
- LLM-as-Judge usa el mismo modelo que el sistema evaluado (sesgo potencial)
- El dataset de evaluación es pequeño (3 casos) — en producción se requieren 100+ casos anotados por expertos de compliance
- Los pesos del retrieval híbrido (α=0.5, β=0.3, γ=0.2) no han sido calibrados empíricamente — son razonados desde papers

### Trabajo futuro para mayor robustez
- Dataset de evaluación anotado por expertos reales de UIAF/CNBV/SBS
- RAGAS para métricas formales de RAG (faithfulness, context_precision)
- Cross-encoder reranker post-retrieval (ms-marco-MiniLM-L-6-v2)
- A/B testing: Claude Haiku vs. Gemini Flash en casos edge de compliance